<a href="https://colab.research.google.com/github/pedroManuelP/C-digos-em-Python/blob/main/Lista3_2_SistControle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [117]:
import numpy as np
from numpy.polynomial import Polynomial
import re

def FT_to_EE(g_num, g_den, s):
  n = g_den.degree() + 1
  alpha = g_den.coef
  beta = g_num.coef
  if(s == '1'):
    A = np.identity(n); A = np.roll(A, -1, axis = 0); A[-1][0] = 0; A[-1,:] = -alpha
    B = np.zeros((n, 1)); B[-1, 0] = 1
    C = np.zeros((1,n)); C[0, 0:(g_num.degree()+1)] = beta
  if(s == '2'):
    A = np.identity(n); A = np.roll(A, -1, axis = 1); A[0][-1] = 0; A[:, -1] = -alpha
    B = np.zeros((n, 1)); B[0:(g_num.degree()+1), 0] = beta
    C = np.zeros((1,n)); C[0, -1] = 1
  return A, B, C

def matrizReferencia(A, B, C):
  Ar = np.zeros((C.shape[0]+A.shape[0], 1+C.shape[1]))
  Ar[0, 1:C.shape[1]+1] = C; Ar[1:A.shape[0]+1, 1:A.shape[1]+1] = A

  Br = np.zeros((B.shape[0]+1, 1)); Br[1:B.shape[0]+1, 0] = B[:, 0]
  return Ar, Br

def matrizExpandida(A, B, C, Kr):
  # Matriz expandida para um seguidor de referência do tipo degrau unitário
  Ae = np.zeros((A.shape[0]+C.shape[0], C.shape[1]+1))
  Ae[0:A.shape[0], 0:B.shape[0]] = A+B*Kr[0, 1:Kr.shape[1]]
  Ae[0:B.shape[0], -1] = (B*Kr[0][0])[:,0]; Ae[-1, 0:C.shape[1]] = C

  Be = np.zeros((Ae.shape[0], 1)); Be[-1, 0] = -1
  Ce = np.zeros((1, Ae.shape[1])); Ce[0, 0:C.shape[1]] = C
  return Ae, Be, Ce

def matrizControlabilidade(A, B):
  n = A.shape[0]
  U = np.zeros((n,n))
  for i in range(n):
    U[:,i:i+1] = (np.linalg.matrix_power(A,i))@B
  posto = np.linalg.matrix_rank(U)
  if(posto == A.shape[0]):
    return U, True
  else:
    return U, False

def realimentaçãoDeEstado(A, U, s):
  theta = Polynomial.fromroots(s)
  #print('\n'+str(theta))
  coeficientes = (theta.coef).real

  n = A.shape[0]; qc = np.zeros((n,n))
  for i in range(n+1):
    qc += coeficientes[i]*(np.linalg.matrix_power(A,i))
  invU = np.linalg.inv(U)
  linha = np.zeros((1,n)); linha[0][-1] = 1
  return -linha@(invU@qc)

In [118]:
# Função G(s)
print("Coeficientes do numerador de G(s): ", end = ''); s = input()
coef_arr = np.array(re.findall(r'-?\d*\.?\d+', s), dtype=float)
g_num = Polynomial(np.flip(coef_arr))
print("Coeficientes do denominador de G(s): ", end = ''); s = input()
coef_arr = np.array(re.findall(r'-?\d*\.?\d+', s), dtype=float)
g_den = Polynomial(np.flip(coef_arr))

# Impressão das funções
print("G(x) = (" + str(g_num) + ")/(" + str(g_den) + ")\n")

# Representação dos Espaços de Estados
print("Como desejar representar a função no espaço de estados?")
print("Digite 1 para forma canônica controlável;"); print("Digite 2 para forma canônica observável.")
s = input(); A, B, C = FT_to_EE(g_num, g_den, s)
print("\nA = " + str(A)); print("B = " + str(B)); print("C = " + str(C))

# Cálculo das matrizes do estado interno
Ar, Br = matrizReferencia(A, B, C)
print("\nAr = " + str(Ar)); print("Br = " + str(Br))

# Cálculo da matriz de controlabilidade
Ur, controlabilidade = matrizControlabilidade(Ar, Br)
if(controlabilidade):
  print("\nA função é controlável, logo é possível fazer um seguidor de referência.")

  # Cálculo da realimentação de estado para o modelo interno
  print("Informe agora os polos:"); s = input(); p = np.zeros((1,0))
  while(s != ''):
    z = np.array(re.findall(r'-?\d*\.?\d+', s), dtype=float);
    if(z.shape[0] == 2):
      p = np.append(p, complex(z[0],z[1]))
    else:
      p = np.append(p, complex(z[0],0))
    s = input();
  print("s = " + str(p))
  Kr = realimentaçãoDeEstado(Ar, Ur, p)
  print("\nKr = " + str(Kr))

  # Cálculo da matriz expandida para um seguidor de referência do tipo degrau unitário
  Ae, Be, Ce = matrizExpandida(A, B, C, Kr)
  print("\nAe = " + str(Ae)); print("Be = " + str(Be)); print("Ce = " + str(Ce))
else:
  print("A função não é controlável, logo não é possível fazer um seguidor de referência.")



Coeficientes do numerador de G(s): 1
Coeficientes do denominador de G(s): 2 2
G(x) = (1.0)/(2.0 + 2.0·x)

Como desejar representar a função no espaço de estados?
Digite 1 para forma canônica controlável;
Digite 2 para forma canônica observável.
1

A = [[ 0.  1.]
 [-2. -2.]]
B = [[0.]
 [1.]]
C = [[1. 0.]]

Ar = [[ 0.  1.  0.]
 [ 0.  0.  1.]
 [ 0. -2. -2.]]
Br = [[0.]
 [0.]
 [1.]]

A função é controlável, logo é possível fazer um seguidor de referência.
Informe agora os polos:
-10
-1-1i
-1+1i

s = [-10.+0.j  -1.-1.j  -1.+1.j]

Kr = [[-20. -20. -10.]]

Ae = [[  0.   1.  -0.]
 [-22. -12. -20.]
 [  1.   0.   0.]]
Be = [[ 0.]
 [ 0.]
 [-1.]]
Ce = [[1. 0. 0.]]
